<a href="https://colab.research.google.com/github/dhikiper/shopsphere-analysis/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

base = "https://raw.githubusercontent.com/dhikiper/shopsphere-analysis/main/"
customers   = pd.read_csv(base + "customers.csv")
products    = pd.read_csv(base + "products.csv")
orders      = pd.read_csv(base + "orders.csv")
order_items = pd.read_csv(base + "order_items.csv")

print(customers.shape, orders.shape, order_items.shape)

(5060, 7) (8803, 8) (15136, 4)


In [4]:
print(customers.isna().sum())
print("Duplicate customers:", customers.duplicated().sum())
print(customers["country"].value_counts())
print(customers["age"].describe())
print(orders.isna().sum())
print("Bad quantities:", (order_items["quantity"] <= 0).sum())

customer_id              0
first_name               0
country                  0
signup_date              0
acquisition_channel      0
age                     40
email                  200
dtype: int64
Duplicate customers: 0
country
United States     1732
Indonesia          800
United Kingdom     738
Germany            637
Canada             621
Australia          472
Name: count, dtype: int64
count    4960.000000
mean       35.577621
std         9.691804
min        18.000000
25%        29.000000
50%        35.000000
75%        42.000000
max        67.000000
Name: age, dtype: float64
order_id            0
customer_id         0
order_date          0
device              0
payment_method      0
status              0
discount_pct       50
shipping_cost     300
dtype: int64
Bad quantities: 25


In [3]:
# Standardize country names
country_map = {
    "usa": "United States", "us": "United States", "united states": "United States",
    "uk": "United Kingdom", "united kingdom": "United Kingdom",
    "germany": "Germany", "indonesia": "Indonesia",
    "australia": "Australia", "canada": "Canada"
}
customers["country"] = customers["country"].str.strip().str.lower().map(country_map)

# Remove duplicates
customers = customers.drop_duplicates(subset="customer_id")

# Invalid ages -> missing
customers.loc[~customers["age"].between(18, 100), "age"] = None

# Convert date
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

print(customers.shape)
print(customers["country"].value_counts())

(5000, 7)
country
United States     1732
Indonesia          800
United Kingdom     738
Germany            637
Canada             621
Australia          472
Name: count, dtype: int64


In [5]:
# Orders
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["discount_missing"] = orders["discount_pct"].isna()
orders["discount_pct"] = orders["discount_pct"].fillna(0)
orders["shipping_cost"] = orders["shipping_cost"].fillna(orders["shipping_cost"].median())

# Order items: drop invalid quantities
order_items = order_items[order_items["quantity"] > 0]

print(orders.isna().sum().sum(), order_items.shape)

0 (15111, 4)


In [6]:
customers.to_csv("clean_customers.csv", index=False)
orders.to_csv("clean_orders.csv", index=False)
order_items.to_csv("clean_order_items.csv", index=False)
products.to_csv("clean_products.csv", index=False)

from google.colab import files
for f in ["clean_customers.csv","clean_orders.csv","clean_order_items.csv","clean_products.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>